In [1]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().parent.resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(
        0,
        str(PROJECT_ROOT),
    )

from src.inference.predict import (
    SleepStagePredictor,
)

**Load final predictor**

In [2]:
predictor = SleepStagePredictor(
    model_path=(
        PROJECT_ROOT
        / "final_model"
        / "stacked_lstm_32_32.keras"
    ),

    scaler_path=(
        PROJECT_ROOT
        / "final_model"
        / "scaler.pkl"
    ),

    config_path=(
        PROJECT_ROOT
        / "final_model"
        / "config.json"
    ),
)

print(
    "Final predictor loaded successfully."
)

Final predictor loaded successfully.


**Load test features**

In [3]:
FEATURES_PATH = (
    PROJECT_ROOT
    / "data"
    / "features"
    / "sleep_edf_features.csv"
)

SPLIT_PATH = (
    PROJECT_ROOT
    / "data"
    / "features"
    / "subject_split.csv"
)

df = pd.read_csv(
    FEATURES_PATH
)

split_df = pd.read_csv(
    SPLIT_PATH
)

df = df.merge(
    split_df,
    on="subject_id",
    how="inner",
)

**Build the test sequences**

In [4]:
from src.preprocessing.normalization import (
    SleepFeatureScaler,
)

from src.preprocessing.sequences import (
    build_sequences_from_dataframe,
)

train_df = (
    df[
        df["split"] == "train"
    ]
    .sort_values(
        ["subject_id", "start_time"]
    )
    .reset_index(drop=True)
)

test_df = (
    df[
        df["split"] == "test"
    ]
    .sort_values(
        ["subject_id", "start_time"]
    )
    .reset_index(drop=True)
)

scaler = SleepFeatureScaler()

X_train_scaled = scaler.fit_transform(
    train_df
)

X_test_scaled = scaler.transform(
    test_df
)

X_test_final, y_test_labels = (
    build_sequences_from_dataframe(
        dataframe=test_df,
        feature_values=X_test_scaled,
        sequence_length=20,
    )
)

print(
    X_test_final.shape
)

(67017, 20, 10)


**Predict the first test sequence**

In [7]:
from src.inference.predict import (
    FEATURE_COLUMNS
)

raw_test_sequence = (
    test_df[
        FEATURE_COLUMNS
    ]
    .iloc[:20]
    .copy()
)

print(
    "Raw sequence shape:",
    raw_test_sequence.shape,
)

print(
    "\nFirst row of raw features:"
)

display(
    raw_test_sequence.head(1)
)

result = predictor.predict_sequence(
    raw_test_sequence
)

print(
    "\nPrediction:"
)

print(
    result
)

Raw sequence shape: (20, 10)

First row of raw features:


,delta_absolute,theta_absolute,alpha_absolute,beta_absolute,delta_relative,theta_relative,alpha_relative,beta_relative,spectral_entropy,dominant_frequency
0,2.063891e-10,2.072891e-11,6.479465e-12,8.046230e-12,0.854105,0.085783,0.026814,0.033298,0.667242,0.585938



Prediction:
{'predicted_stage': 'W', 'predicted_index': 0, 'probabilities': {'W': 0.999996542930603, 'N1': 2.3999764380278066e-06, 'N2': 7.816909715074871e-07, 'N3': 3.5433731504497246e-09, 'REM': 2.1721105269989494e-07}}


**Batch prediction test**

In [8]:
predictions, probabilities = (
    predictor.predict_batch(
        X_test_final[:100]
    )
)

print(
    "Predictions shape:",
    predictions.shape,
)

print(
    "Probabilities shape:",
    probabilities.shape,
)

print(
    "First 10 predictions:"
)

print(
    predictions[:10]
)

Predictions shape: (100,)
Probabilities shape: (100, 5)
First 10 predictions:
[0 0 0 0 0 0 0 0 0 0]


**Batch prediction using ALREADY SCALED sequences**

In [9]:
predictions, probabilities = (
    predictor.predict_batch(
        X_test_final[:100]
    )
)

print(
    "Predictions shape:",
    predictions.shape,
)

print(
    "Probabilities shape:",
    probabilities.shape,
)

Predictions shape: (100,)
Probabilities shape: (100, 5)


**Verify the inference API**

In [10]:
from src.inference.predict import (
    SleepStagePredictor,
)

In [11]:
predictor = SleepStagePredictor(
    model_path=(
        PROJECT_ROOT
        / "final_model"
        / "stacked_lstm_32_32.keras"
    ),
    scaler_path=(
        PROJECT_ROOT
        / "final_model"
        / "scaler.pkl"
    ),
    config_path=(
        PROJECT_ROOT
        / "final_model"
        / "config.json"
    ),
)

print(
    "Predictor loaded successfully."
)

Predictor loaded successfully.


In [12]:
import numpy as np

raw_sequences = np.stack(
    [
        test_df[
            FEATURE_COLUMNS
        ]
        .iloc[
            i:i + 20
        ]
        .to_numpy(
            dtype=np.float32
        )
        for i in [0, 20, 40]
    ],
    axis=0,
)

print(
    "Raw batch shape:",
    raw_sequences.shape,
)

Raw batch shape: (3, 20, 10)


In [13]:
predictions, probabilities = (
    predictor.predict_batch(
        raw_sequences
    )
)

print(
    "Predictions:",
    predictions
)

print(
    "Predictions shape:",
    predictions.shape
)

print(
    "Probabilities shape:",
    probabilities.shape
)

Predictions: [0 0 0]
Predictions shape: (3,)
Probabilities shape: (3, 5)
